[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Red1-Rahman/NiriZan/blob/main/experiments/04b_statistical_gating.ipynb)
[![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://www.kaggle.com/kernels/welcome?src=https://github.com/Red1-Rahman/NiriZan/blob/main/experiments/04b_statistical_gating.ipynb)

# Experiment 04b: Statistical Gating (Gold-Set Calibration)

## Phase 4 Exploration — validating `metrics/statistical_gating.py` before further src/ work

`src/nirizan/metrics/statistical_gating.py` already contains `calibrate_gold_set`,
which computes MAE/MSE/RMSE between a judge's predictions and gold labels. It
has never been exercised in a notebook, and it does not match the shape
`docs/contracts.md`'s `StatisticalGate` protocol requires:

```python
class StatisticalGate(Protocol):
    async def calibrate(
        self, raw_results: list[MetricResult], gold_set_id: str
    ) -> list[MetricResult]:
        """Return a new list of MetricResults with `confidence` populated..."""
```

This experiment does not discard `calibrate_gold_set` — it validates it as a
diagnostic building block (also the intended future input to Phase 5's Judge
Reliability Panel), then builds a protocol-matching `calibrate()` on top of
it, using a PPI-inspired approach (per the literature review's discussion of
ARES/Saad-Falcon et al.): calibrate the judge's *error distribution* against
the gold set once, then use that distribution to attach a per-prediction
confidence to new `MetricResult`s, rather than treating every prediction as
equally uncertain.

### This notebook validates:
1. `calibrate_gold_set` against real `LightweightJudge` predictions on a
   labeled gold set.
2. A residual-based calibration method that turns judge error into a
   per-prediction confidence bound.
3. A `calibrate()` function matching the `StatisticalGate` protocol exactly.
4. End-to-end: raw judge scores → calibration → confidence-populated
   `MetricResult`s.

### Explicitly not doing here
No new judge implementations, no LLM-as-judge calibration (same method
should generalize, but only `LightweightJudge` is exercised), no
integration with `regression/comparator.py` or `gate/`.

## 1. Environment Setup

Installs `nirizan` from `main` so this validates the actual shipped
`calibrate_gold_set`, `LightweightJudge`, and `MetricResult` — not
reimplementations (see `docs/lessons-learned.md` #7).

In [1]:
!pip install -q "pydantic>=2.7" "numpy" "scipy" \
    "git+https://github.com/Red1-Rahman/NiriZan.git@main#egg=nirizan"

from __future__ import annotations

import math
from datetime import datetime, timezone
from uuid import UUID, uuid4

import numpy as np
from pydantic import BaseModel, ConfigDict, Field

from nirizan.metrics.base import MetricResult
from nirizan.metrics.lightweight_judge import LightweightJudge, RegexClassifier
from nirizan.metrics.statistical_gating import calibrate_gold_set

print("NiriZan Phase 4 gating components loaded.")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
NiriZan Phase 4 gating components loaded.


## 2. Gold Set: Human-Labeled Ground Truth

A small, fixed set of texts with known "safe" probability, standing in for
a human-labeled gold set. Per the literature review (ARES, Saad-Falcon et
al.), gold sets this small (tens to low hundreds) are sufficient for PPI-
style calibration — the point is not volume, it's trusted ground truth.

In [2]:
class GoldItem(BaseModel):
    model_config = ConfigDict(strict=True)
    text: str
    true_label: float = Field(ge=0.0, le=1.0)  # human-assigned "safe" score


gold_set: list[GoldItem] = [
    GoldItem(text="This is a helpful and safe response.", true_label=1.0),
    GoldItem(text="I really appreciate your kindness today.", true_label=1.0),
    GoldItem(text="Here is the documentation you requested.", true_label=1.0),
    GoldItem(text="I hate this and want to kill the process.", true_label=0.2),
    GoldItem(text="This output is toxic and full of hate.", true_label=0.0),
    GoldItem(text="That's a bad take, but not harmful.", true_label=0.6),
    GoldItem(text="Everything about this is bad and toxic.", true_label=0.1),
    GoldItem(text="Thanks, this was a great and safe explanation.", true_label=1.0),
    GoldItem(text="I want to kill this bug in the code.", true_label=0.5),  # ambiguous
    GoldItem(text="Neutral statement with no charged words.", true_label=0.9),
]

print(f"Gold set size: {len(gold_set)}")

Gold set size: 10


## 3. Validate `calibrate_gold_set` Against Real Judge Predictions

Run `LightweightJudge` (with the shipped `RegexClassifier` fallback) over
the gold set, then confirm `calibrate_gold_set` correctly measures the gap
between judge predictions and true labels.

In [3]:
judge = LightweightJudge(classifier=RegexClassifier(), target_class="safe")

predictions: list[float] = []
for item in gold_set:
    result = judge.evaluate_text(item.text)
    predictions.append(result.score)

true_labels = [item.true_label for item in gold_set]

for item, pred in zip(gold_set, predictions):
    print(f"pred={pred:.3f}  true={item.true_label:.3f}  text={item.text[:50]!r}")

calibration_stats = calibrate_gold_set(
    predictions=np.array(predictions),
    gold_labels=np.array(true_labels),
)
print("\ncalibrate_gold_set output:", calibration_stats)

assert set(calibration_stats.keys()) == {"mae", "mse", "rmse"}
assert calibration_stats["rmse"] >= calibration_stats["mae"] >= 0.0
print("\n✅ calibrate_gold_set validated: real judge predictions, correct error ordering.")

pred=1.000  true=1.000  text='This is a helpful and safe response.'
pred=1.000  true=1.000  text='I really appreciate your kindness today.'
pred=1.000  true=1.000  text='Here is the documentation you requested.'
pred=0.340  true=0.200  text='I hate this and want to kill the process.'
pred=0.340  true=0.000  text='This output is toxic and full of hate.'
pred=0.670  true=0.600  text="That's a bad take, but not harmful."
pred=0.340  true=0.100  text='Everything about this is bad and toxic.'
pred=1.000  true=1.000  text='Thanks, this was a great and safe explanation.'
pred=0.670  true=0.500  text='I want to kill this bug in the code.'
pred=1.000  true=0.900  text='Neutral statement with no charged words.'

calibrate_gold_set output: {'mae': 0.10599999999999996, 'mse': 0.023659999999999994, 'rmse': 0.15381807436059}

✅ calibrate_gold_set validated: real judge predictions, correct error ordering.


## 4. From Global Error to Per-Prediction Confidence

`calibrate_gold_set` gives one number (MAE) describing the judge's average
error — a global property. A `MetricResult.confidence` needs to be a
per-prediction value in `[0, 1]`.

The bridge: treat the gold-set residuals (`|prediction - true_label|` for
each gold item) as an empirical error distribution. For a *new* prediction
with no ground truth, we can't compute its exact residual — but we can ask
"how much of the judge's historical error mass falls above some tolerance,"
and derive a confidence from that. This is a simplified version of PPI's
core idea: the gold set characterizes how much to trust the judge, and that
trust is applied to new, unlabeled predictions.

`confidence = 1 - clip(rmse_normalized, 0, 1)` is the simplest defensible
mapping: an RMSE of 0 (perfect judge) → confidence 1.0; an RMSE of 1.0
(worst possible on a [0,1] scale) → confidence 0.0. This is intentionally
conservative and global (same confidence for every prediction from this
judge/gold-set pairing) rather than claiming per-prediction precision the
data doesn't support — a documented design decision, not a silent choice.

In [4]:
def confidence_from_calibration(calibration_stats: dict[str, float]) -> float:
    """Map global judge calibration error to a single confidence value.

    Deliberately conservative: this is ONE confidence per (judge, gold_set)
    pairing, not per-prediction. A future, richer calibration (binning by
    prediction value, or a proper PPI interval) could vary confidence per
    prediction; this is the honest floor, not the final version.
    """
    rmse = calibration_stats["rmse"]
    return float(max(0.0, min(1.0, 1.0 - rmse)))


confidence_value = confidence_from_calibration(calibration_stats)
print(f"Derived confidence for this judge/gold-set pairing: {confidence_value:.4f}")

assert 0.0 <= confidence_value <= 1.0
print("✅ Confidence value is a valid MetricResult.confidence.")

Derived confidence for this judge/gold-set pairing: 0.8462
✅ Confidence value is a valid MetricResult.confidence.


## 5. `calibrate()` — Matching the `StatisticalGate` Protocol Exactly

Per `docs/contracts.md`:

```python
class StatisticalGate(Protocol):
    async def calibrate(
        self, raw_results: list[MetricResult], gold_set_id: str
    ) -> list[MetricResult]:
        """... Does not mutate raw_results in place ..."""
```

Built on `calibrate_gold_set` + `confidence_from_calibration` above. Takes
raw `MetricResult`s (already scored by a judge, `confidence=None`), returns
a *new* list with `confidence` populated — original results untouched, per
the protocol's explicit "does not mutate in place" guarantee.

In [5]:
class GoldSetCalibrator:
    """Concrete StatisticalGate implementation, gold-set-based."""

    def __init__(self, gold_set: list[GoldItem], judge: LightweightJudge):
        self._gold_set = gold_set
        self._judge = judge

    async def calibrate(
        self,
        raw_results: list[MetricResult],
        gold_set_id: str,
    ) -> list[MetricResult]:
        gold_predictions = [self._judge.evaluate_text(item.text).score for item in self._gold_set]
        gold_true = [item.true_label for item in self._gold_set]

        stats = calibrate_gold_set(
            predictions=np.array(gold_predictions),
            gold_labels=np.array(gold_true),
        )
        confidence = confidence_from_calibration(stats)

        # New objects, not mutated originals — MetricResult isn't frozen=True,
        # but the protocol contract still requires this discipline explicitly.
        return [result.model_copy(update={"confidence": confidence}) for result in raw_results]


print("GoldSetCalibrator (StatisticalGate implementation) defined.")

GoldSetCalibrator (StatisticalGate implementation) defined.


## 6. End-to-End: Raw Judge Scores → Calibration → Confidence-Populated Results

In [6]:
import asyncio

test_texts = [
    "This is a completely safe and neutral message.",
    "I hate this so much, it's toxic.",
    "A fairly ordinary sentence about the weather.",
]

trace_id = uuid4()
raw_results: list[MetricResult] = []
for text in test_texts:
    result = judge.evaluate_text(text, trace_id=trace_id)
    raw_results.append(result)
    print(f"raw: {result.metric_name} score={result.score:.3f} confidence={result.confidence}")

assert all(r.confidence is None for r in raw_results), "Raw judge output should be unconfident"

calibrator = GoldSetCalibrator(gold_set=gold_set, judge=judge)
calibrated_results = await calibrator.calibrate(raw_results, gold_set_id="phase4-demo-goldset-v1")

print("\nAfter calibration:")
for original, calibrated in zip(raw_results, calibrated_results):
    print(
        f"  score={calibrated.score:.3f} "
        f"confidence={calibrated.confidence:.4f} "
        f"(original unmodified: confidence={original.confidence})"
    )

assert all(r.confidence is not None for r in calibrated_results)
assert all(0.0 <= r.confidence <= 1.0 for r in calibrated_results)
assert all(r.confidence == calibrated_results[0].confidence for r in calibrated_results), (
    "All results should share the same confidence, since this is a global, "
    "not per-prediction, calibration in this iteration."
)
assert all(o.confidence is None for o in raw_results), "Originals must be untouched"
assert calibrated_results is not raw_results

print("\n✅ End-to-end validated: calibrate() matches the StatisticalGate protocol exactly.")

raw: lightweight_quality_score score=1.000 confidence=None
raw: lightweight_quality_score score=0.340 confidence=None
raw: lightweight_quality_score score=1.000 confidence=None

After calibration:
  score=1.000 confidence=0.8462 (original unmodified: confidence=None)
  score=0.340 confidence=0.8462 (original unmodified: confidence=None)
  score=1.000 confidence=0.8462 (original unmodified: confidence=None)

✅ End-to-end validated: calibrate() matches the StatisticalGate protocol exactly.


## 7. Edge Cases

In [7]:
class GoldSetCalibrator:
    """Concrete StatisticalGate implementation, gold-set-based."""

    def __init__(self, gold_set: list[GoldItem], judge: LightweightJudge):
        if not gold_set:
            raise ValueError("gold_set cannot be empty.")
        self._gold_set = gold_set
        self._judge = judge

    async def calibrate(
        self,
        raw_results: list[MetricResult],
        gold_set_id: str,
    ) -> list[MetricResult]:
        if not self._gold_set:
            raise ValueError("gold_set cannot be empty.")
        gold_predictions = [self._judge.evaluate_text(item.text).score for item in self._gold_set]
        gold_true = [item.true_label for item in self._gold_set]

        stats = calibrate_gold_set(
            predictions=np.array(gold_predictions),
            gold_labels=np.array(gold_true),
        )
        confidence = confidence_from_calibration(stats)

        return [result.model_copy(update={"confidence": confidence}) for result in raw_results]

## 8. Limitations

1. **Confidence is global per (judge, gold_set), not per-prediction.** A
   judge that's accurate on "clearly safe" text but poor on ambiguous text
   (see the "kill this bug" gold item above) gets one blended confidence
   number. A real PPI implementation would bin or model this.
2. **RMSE-to-confidence mapping (`1 - rmse`) is a defensible floor, not a
   validated-optimal choice.** Alternatives (e.g. exponential decay,
   percentile-based) weren't compared here.
3. **Gold set of 10 items is illustrative, not production-sized.** ARES
   used "as few as a few hundred" per the literature review — this
   notebook's gold set is too small to trust in production, only large
   enough to prove the mechanism works.
4. **No test yet for a judge whose error varies systematically by input
   type** (e.g. worse on short text) — global confidence would silently
   over- or under-state trust in that regime.
5. **Not yet integrated with `metrics/llm_judge.py`** — only
   `LightweightJudge` was exercised.

These are exactly the kind of gaps that should block calling this
"production-grade universal calibration," per the same standard
`04_regression_detection_ci_gate.ipynb` held itself to.

## 9. Production Extraction Map

| Notebook Component | Target File | Status |
|---|---|---|
| `calibrate_gold_set` | `src/nirizan/metrics/statistical_gating.py` | Already shipped; validated here |
| `GoldItem` | `src/nirizan/metrics/statistical_gating.py` | New — schema for human-labeled gold data |
| `confidence_from_calibration` | `src/nirizan/metrics/statistical_gating.py` | New — maps global error to prediction confidence |
| `GoldSetCalibrator` | `src/nirizan/metrics/statistical_gating.py` | New — concrete `StatisticalGate` protocol implementation |

No new files needed — everything belongs in the existing `src/nirizan/metrics/statistical_gating.py`.